In [5]:
%load_ext autoreload
%autoreload 2

import util as yu
from util import *
import util_moments as yum

yu.setpath('analysis_xJ_syst')

folder='step1'
fit_conn=['2st','sum'][0]
fit_Q2_conn=['Q2fit','0Q2'][0]; fit_Q2_dipole_Q2cut=1; fit_Q2_const_Q2cut=0.5
fit_discq=fit_discg=['const','band','sum'][0]

enss=['b','c','d','e']
stouts_all=[5,7,10,13,15,20]
stouts=stouts_all
tfphys=[0.4,0.5,0.6,0.7,0.8,0.9]
tcphys=[0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4]
tcphys=[0.1,0.2,0.3,0.4,0.5]
tftcphys=[(tfphy,tcphy) for tfphy in tfphys for tcphy in tcphys if tfphy>=2*tcphy]

path='data_aux/analysis_xJ_syst_input_step1.json'
dic=yu.load_json(path)
if fit_conn in ['2st']:
    tftcphy_A20_conn=tuple(dic['tftcphy_A20_conn'])
elif fit_conn in ['sum']:
    tftcphy_A20_conn=(dic['tftcphy_A20_conn_sum'][0],tcphys[0])
else:
    1/0
    
if fit_discq in ['const','band']:
    tftcphy_A20_discq=tuple(dic['tftcphy_A20_discq']) 
elif fit_discq in ['sum']:
    tftcphy_A20_discq=(dic['tftcphy_A20_discq_sum'][0],tcphys[0])
else:
    1/0
    
if fit_discg in ['const','band']:
    nst2tftcphy_A20_discg={nst:tuple(dic[f'tftcphy_A20_discg'][str(nst)]) if str(nst) in dic[f'tftcphy_A20_discg'] else tuple(dic[f'tftcphy_A20_discg']['default']) for nst in stouts}
elif fit_discg in ['sum']:
    nst2tftcphy_A20_discg={nst:(dic[f'tftcphy_A20_discg_sum'][str(nst)][0],tcphys[0]) if str(nst) in dic[f'tftcphy_A20_discg_sum'] else (dic[f'tftcphy_A20_discg_sum']["default"][0],tcphys[0]) for nst in stouts}
else:
    1/0
    
stouts_jointlinear=dic['stouts_jointlinear']

js_conn=['j+;conn','j-;conn']
js_discq=['j+;disc','js;disc','jc;disc'] 
js_gluon=[f'jg;stout{nst}' for nst in stouts]
js_disc=js_discq+js_gluon


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
ens2Njk={'b':725,'c':400,'d':493,'e':516}

path='data_aux/RCs_np.pkl'
with open(path,'rb') as f:
    ens2RCs_np_me=pickle.load(f)
ens2RCs_np=yu.load_pkl_reg('ens2RCs_np_jk',pathlabel='processRenorms')

pdflabels=['HERAPDF2.0','ABMP16','CT18','MSHT20','NNPDF4.0','PDF4LHC21','JAM22','CJ22']
pdfsets=['HERAPDF20_NNLO_EIG','ABMP16_4_nnlo','CT18NNLO','MSHT20nnlo_as118','NNPDF40_nnlo_as_01180','PDF4LHC21_40_pdfas','JAM22-PDF_proton_nlo','CJ22']
label2j2me_A20=yu.load_pkl_reg('label2j2me_avgx',pathlabel='lhapdf')
label2j2me_A20={label:{j[:-2] if j.endswith(';+') else j:me for j,me in label2j2me_A20[label].items()} for label in label2j2me_A20.keys()}

# A20

In [7]:
def run():
    cttejn_key2bare_A20={}

    for ens in enss:
        for j in js_conn+js_disc:
            if j in js_conn:
                app={'2st':'2st_True','sum':'sum'}[fit_conn]
            elif j in js_discq:
                app={'const':'const','band':'band','sum':'sum'}[fit_discq]
            elif j in js_gluon:
                app={'const':'const','band':'band','sum':'sum'}[fit_discg]
            else:
                1/0
            fits=yu.getFits(f'{ens}_{j}_{app}',pathlabel='analysis_avgx_2')
            fitlabels=[fit[0] for fit in fits]

            for tfphy,tcphy in tftcphys:
                if app in ['sum']:
                    ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + (0 if tcmin==2 else 999) for tfmin,tcmin in fitlabels])
                else:
                    ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs(tcmin*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
                fit=fits[ind]
                # if ens=='e':
                #     print(tfphy,tcphy,fit[0])
                
                if j in js_conn:
                    key=('equal',tfphy,tcphy,ens,j,(0,0,0))
                    cttejn_key2bare_A20[key]=fit[1][:,0]
                    key=('all',tfphy,tcphy,ens,j,(0,0,0))
                    cttejn_key2bare_A20[key]=fit[1][:,0]
                else:
                    if app in ['const','sum']:
                        t=fit[1][:,0]
                    elif app=='band':
                        gett=lambda t:round(t/yu.ens2a[ens])
                        t=yu.doWA_band(fits,tf_min=gett(tfphy),tf_max=gett(1.),tcmin=max(2,gett(tcphy)),corrQ=False)
                        t=t[0][:,0]                    
                    key=('unequal',tfphy,tcphy,ens,j,(0,1,1))
                    cttejn_key2bare_A20[key]=t

    def decodeTask(task):
        n2qpp1,ff,j=task.split('_')
        j=j.replace(',',';')
        n2qpp1=tuple([int(ele) for ele in n2qpp1.split(',')])
        return (n2qpp1,ff,j)
    path='data_aux/dat_ignore/analysis_A20_conn_run'
    with open(path,'r') as f:
        tasks=f.read().splitlines()
    for ens in enss:
        for j in js_conn:
            for c1 in ['all','unequal','equal']:
                for task in tasks:
                    (n2qpp1,ff,j)=decodeTask(task)
                    
                    app={'2st':'2st','sum':'sum'}[fit_conn]
                    fits=yu.getFits(f'{n2qpp1}_{ff}_{j}_{ens}_{c1}_err_{app}',pathlabel='analysis_A20')
                    if fits is None:
                        continue
                    fitlabels=[fit[0] for fit in fits]
                    
                    for tfphy,tcphy in tftcphys:
                        if app in ['sum']:
                            ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + (0 if tcmin==(2,2) else 999) for tfmin,tcmin in fitlabels])
                        else:
                            ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs((tcmin[0]+tcmin[1])/2*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
                        fit=fits[ind]
                                
                        key=(c1,tfphy,tcphy,ens,j,n2qpp1)
                        cttejn_key2bare_A20[key]=fit[1][:,0]
                        
    path='data_aux/dat_ignore/analysis_A20_disc_run'
    with open(path,'r') as f:
        tasks=f.read().splitlines()
    for ens in enss:
        for j in js_disc:
            for c1 in ['unequal']:
                for task in tasks:
                    (n2qpp1,ff,j)=decodeTask(task)
                    
                    if j in js_discq:
                        app={'const':'const','band':'band','sum':'sum'}[fit_discq]
                    elif j in js_gluon:
                        app={'const':'const','band':'band','sum':'sum'}[fit_discg]
                    else:
                        1/0
                    
                    fits=yu.getFits(f'{n2qpp1}_{ff}_{j}_{ens}_{c1}_err_{app}',pathlabel='analysis_A20')
                    if fits is None:
                        continue
                    fitlabels=[fit[0] for fit in fits]
                    
                    for tfphy,tcphy in tftcphys:
                        if app in ['sum']:
                            ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + (0 if tcmin==(2,2) else 999) for tfmin,tcmin in fitlabels])
                        else:
                            ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs((tcmin[0]+tcmin[1])/2*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
                        fit=fits[ind]
                        
                        key=(c1,tfphy,tcphy,ens,j,n2qpp1)
                        cttejn_key2bare_A20[key]=fit[1][:,0]
    
    return cttejn_key2bare_A20

cttejn_key2bare_A20=yu.load_pkl_reg(f'{folder}/cttejn_key2bare_A20')
if cttejn_key2bare_A20 is None: 
    cttejn_key2bare_A20=run()
    yu.save_pkl_reg(f'{folder}/cttejn_key2bare_A20',cttejn_key2bare_A20,mkdirQ=True)

In [8]:
def run():
    ttej_key2bare_A20at0={}
    
    for ind,tftcphy in enumerate(tftcphys):
        print(f'{ind}/{len(tftcphys)}',tftcphy,end='          \r')
        colors=['r','g','b','orange']
        for j in js_conn+js_disc:
            c1s=['unequal','all','equal'] if j in js_conn else ['unequal']
            
            fig,axs=yu.getFigAxs(len(enss)+1,1+len(c1s)+1,Lrow=4,Lcol=8,sharex='col',sharey=True)
            fig.suptitle([f'{c}={c1}' for c,c1 in zip(colors,c1s)])
            yu.addRowHeader(axs,enss)
            yu.addColHeader(axs,['compare']+c1s+['compare at Q2=0'])
            
            for axr in axs:
                for ax in axr:
                    yu.addRefLine(ax,0)

            for ic1,c1 in enumerate(c1s):
                key2bare={cttejn[3:]:cttejn_key2bare_A20[cttejn] for cttejn in cttejn_key2bare_A20.keys() if cttejn[:3]==(c1,tftcphy[0],tftcphy[1])}
                for iens,ens in enumerate(enss):
                    
                    if c1=='unequal' and j in js_disc:
                        t=cttejn_key2bare_A20[('unequal',tftcphy[0],tftcphy[1],ens,j,(0,1,1))]
                        ttej_key2bare_A20at0[(tftcphy[0],tftcphy[1],ens,j)]=t
                        continue
                    
                    ax=axs[iens,0]            
                    xunit=1; yunit=1
                    
                    if j in js_conn:
                        Zqq='Zqq^s' if j in ['j+;conn'] else 'Zqq'
                        mn='mu=nu' if c1 in ['all','equal'] else 'mu!=nu'
                        Z=ens2RCs_np_me[ens][f'{Zqq}({mn})']
                    else:
                        Z=1
                    
                    n2qpp1s=[key[-1] for key in key2bare.keys() if key[0]==ens and key[1]==j]
                    Q2s=np.array([yum.n2qpp12Q2(n2qpp1,ens) for n2qpp1 in n2qpp1s])
                    ys=np.array([key2bare[(ens,j,n2qpp1)][:] for n2qpp1 in n2qpp1s]).T
                    
                    if ic1==0:
                        if j in js_conn:
                            m,e=yu.jackme(cttejn_key2bare_A20[('equal',tftcphy[0],tftcphy[1],ens,j,(0,0,0))]*ens2RCs_np_me[ens][f'{Zqq}(mu=nu)'])
                        else:
                            m,e=yu.jackme(cttejn_key2bare_A20[('unequal',tftcphy[0],tftcphy[1],ens,j,(0,1,1))])
                        axs[iens,-1].errorbar(-1,m,e,color='orange',label=yu.un2str(m,e))
                    
                    color=colors[ic1]
                    
                    mean,err=yu.jackme(ys*Z)
                    plt_x=(Q2s+(ic1-1)*0.005)*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
                    ax.errorbar(plt_x,plt_y,plt_yerr,color=color)
                    axs[iens,ic1+1].errorbar(plt_x,plt_y,plt_yerr,color=color)
                    axs[-1,ic1+1].errorbar(plt_x,plt_y,plt_yerr,color=colors[iens])
                    
                    
                    if j in js_conn:
                        Q2cut=fit_Q2_dipole_Q2cut
                        doFit=yu.doFit_dipole
                        fitfunc=yu.fitfunc_dipole
                    else:
                        Q2cut=fit_Q2_const_Q2cut
                        doFit=yu.doFit_linear
                        fitfunc=yu.fitfunc_linear
                    inds=[i for i,Q2 in enumerate(Q2s) if Q2<=Q2cut]
                    Q2s=Q2s[inds]
                    ys=ys[:,inds]
                
                    pars_jk,chi2_jk,Ndof=doFit(Q2s,ys,corrQ=False)
                    
                    xs_plt=np.arange(0,Q2cut+1e-5,0.05)
                    y_plt=np.array([fitfunc(xs_plt,*pars) for pars in pars_jk])
                    mean,err=yu.jackme(y_plt*Z)
                    plt_x=xs_plt*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
                    
                    axs[iens,ic1+1].fill_between(plt_x,plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2,label=yu.un2str(plt_y[0],plt_yerr[0]))
                    axs[iens,ic1+1].legend()
                    
                    axs[iens,-1].errorbar(ic1,plt_y[0],plt_yerr[0],color=color,label=yu.un2str(plt_y[0],plt_yerr[0]))
                    axs[iens,-1].set_xlim([-5,5])
                    axs[iens,-1].legend()
                    
                    if True:
                        if c1=='all' and j in ['j-;conn']:
                            ttej_key2bare_A20at0[(tftcphy[0],tftcphy[1],ens,j+'_0Q2')]=cttejn_key2bare_A20[('equal',tftcphy[0],tftcphy[1],ens,j,(0,0,0))]*ens2RCs_np_me[ens][f'Zqq(mu=nu)']
                        if j in ['j-;conn']:
                            mn='mu=nu' if c1 in ['all','equal'] else 'mu!=nu'
                            t= y_plt[:,0] * ens2RCs_np_me[ens][f'Zqq({mn})']
                            ttej_key2bare_A20at0[(tftcphy[0],tftcphy[1],ens,j+'_'+c1)]=t
                            
                    if fit_Q2_conn=='Q2fit':
                        if c1=='unequal' and j in ['j+;conn']:
                            t=y_plt[:,0]
                            ttej_key2bare_A20at0[(tftcphy[0],tftcphy[1],ens,j)]=t
                        if c1=='all' and j in ['j-;conn']:
                            factor=ens2RCs_np_me[ens][f'Zqq(mu=nu)']/ens2RCs_np_me[ens][f'Zqq(mu!=nu)']
                            t=y_plt[:,0]*factor
                            ttej_key2bare_A20at0[(tftcphy[0],tftcphy[1],ens,j)]=t
                    elif fit_Q2_conn=='0Q2':
                        if c1=='all' and j in js_conn:
                            factor=ens2RCs_np_me[ens][f'Zqq(mu=nu)']/ens2RCs_np_me[ens][f'Zqq(mu!=nu)']
                            t=cttejn_key2bare_A20[('equal',tftcphy[0],tftcphy[1],ens,j,(0,0,0))]*factor
                            ttej_key2bare_A20at0[(tftcphy[0],tftcphy[1],ens,j)]=t
                            
                    if c1=='unequal' and j in js_disc:
                        t=cttejn_key2bare_A20[('unequal',tftcphy[0],tftcphy[1],ens,j,(0,1,1))]
                        ttej_key2bare_A20at0[(tftcphy[0],tftcphy[1],ens,j)]=t

            yu.finalizePlot(f'{folder}/Q2fit_A20/{tftcphy}/{j}',closeQ=True,mkdirQ=True)
            yu.finalizePlot(closeQ=True)
            
    return ttej_key2bare_A20at0

ttej_key2bare_A20at0=yu.load_pkl_reg(f'{folder}/ttej_key2bare_A20at0')
if ttej_key2bare_A20at0 is None:
    ttej_key2bare_A20at0=run()
    yu.save_pkl_reg(f'{folder}/ttej_key2bare_A20at0',ttej_key2bare_A20at0)

# B20

In [9]:
def run():
    cttejn_key2bare_B20={}
    
    def decodeTask(task):
        n2qpp1,ff,j=task.split('_')
        j=j.replace(',',';')
        n2qpp1=tuple([int(ele) for ele in n2qpp1.split(',')])
        return (n2qpp1,ff,j)
    path='data_aux/dat_ignore/analysis_B20_conn_run'
    with open(path,'r') as f:
        tasks=f.read().splitlines()
    for ens in enss:
        for j in js_conn:
            app={'2st':'2st','sum':'sum'}[fit_conn]
            
            for c1 in ['all','unequal','equal']:
                for task in tasks:
                    (n2qpp1,ff,j)=decodeTask(task)
                    
                    fits=yu.getFits(f'{n2qpp1}_{ff}_{j}_{ens}_{c1}_err_{app}',pathlabel='analysis_B20')
                    if fits is None:
                        continue
                    fitlabels=[fit[0] for fit in fits]
                    
                    for tfphy,tcphy in tftcphys:
                        if app in ['sum']:
                            ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + (0 if tcmin==(2,2) else 999) for tfmin,tcmin in fitlabels])
                        else:
                            ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs((tcmin[0]+tcmin[1])/2*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
                            
                        fit=fits[ind]
                        
                        key=(c1,tfphy,tcphy,ens,j,n2qpp1)
                        cttejn_key2bare_B20[key]=fit[1][:,0]
                        
    path='data_aux/dat_ignore/analysis_B20_disc_run'
    with open(path,'r') as f:
        tasks=f.read().splitlines()
    for ens in enss:
        for j in js_disc:
            if j in js_discq:
                app={'const':'const','band':'band','sum':'sum'}[fit_discq]
            elif j in js_gluon:
                app={'const':'const','band':'band','sum':'sum'}[fit_discg]
            else:
                1/0
            for c1 in ['unequal']:
                for task in tasks:
                    (n2qpp1,ff,j)=decodeTask(task)
                    
                    fits=yu.getFits(f'{n2qpp1}_{ff}_{j}_{ens}_{c1}_err_{app}',pathlabel='analysis_B20_2')
                    if fits is None:
                        continue
                    fitlabels=[fit[0] for fit in fits]
                    
                    for tfphy,tcphy in tftcphys:
                        if app in ['sum']:
                            ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + (0 if tcmin==(2,2) else 999) for tfmin,tcmin in fitlabels])
                        else:
                            ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs((tcmin[0]+tcmin[1])/2*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
                        fit=fits[ind]
                        
                        if app in ['const','sum']:
                            t=fit[1][:,0]
                        elif app=='band':
                            gett=lambda t:round(t/yu.ens2a[ens])
                            t=yu.doWA_band(fits,tf_min=gett(tfphy),tf_max=gett(1.3),tcmin=max(2,gett(tcphy)),corrQ=False)
                            t=t[0][:,0]    
                        
                        key=(c1,tfphy,tcphy,ens,j,n2qpp1)
                        cttejn_key2bare_B20[key]=fit[1][:,0]
    return cttejn_key2bare_B20

cttejn_key2bare_B20=yu.load_pkl_reg(f'{folder}/cttejn_key2bare_B20')
if cttejn_key2bare_B20 is None: 
    cttejn_key2bare_B20=run()
    yu.save_pkl_reg(f'{folder}/cttejn_key2bare_B20',cttejn_key2bare_B20,mkdirQ=True)

In [10]:
def run():
    ttej_key2bare_B20at0={}
    
    for ind,tftcphy in enumerate(tftcphys):
        print(f'{ind}/{len(tftcphys)}',tftcphy,end='          \r')
        
        colors=['r','g','b','orange']
        for j in js_conn+js_disc:
            c1s=['unequal','all','equal'] if j in js_conn else ['unequal']
            
            fig,axs=yu.getFigAxs(len(enss)+1,1+len(c1s)+1,Lrow=4,Lcol=8,sharex='col',sharey=True)
            fig.suptitle([f'{c}={c1}' for c,c1 in zip(colors,c1s)])
            yu.addRowHeader(axs,enss)
            yu.addColHeader(axs,['compare']+c1s+['compare at Q2=0'])
            
            for axr in axs:
                for ax in axr:
                    yu.addRefLine(ax,0)

            for ic1,c1 in enumerate(c1s):
                key2bare={cttejn[3:]:cttejn_key2bare_B20[cttejn] for cttejn in cttejn_key2bare_B20.keys() if cttejn[:3]==(c1,tftcphy[0],tftcphy[1])}
                for iens,ens in enumerate(enss):
                    ax=axs[iens,0]            
                    xunit=1; yunit=1
                    
                    if j in js_conn:
                        Zqq='Zqq^s' if j in ['j+;conn'] else 'Zqq'
                        mn='mu=nu' if c1 in ['all','equal'] else 'mu!=nu'
                        Z=ens2RCs_np_me[ens][f'{Zqq}({mn})']
                    else:
                        Z=1
                    
                    n2qpp1s=[key[-1] for key in key2bare.keys() if key[0]==ens and key[1]==j]
                    Q2s=np.array([yum.n2qpp12Q2(n2qpp1,ens) for n2qpp1 in n2qpp1s])
                    ys=np.array([key2bare[(ens,j,n2qpp1)][:] for n2qpp1 in n2qpp1s]).T

                    color=colors[ic1]
                    
                    # print(tftcphy,j,Q2s.shape,ys.shape)
                    
                    mean,err=yu.jackme(ys*Z)
                    plt_x=(Q2s+(ic1-1)*0.005)*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
                    ax.errorbar(plt_x,plt_y,plt_yerr,color=color)
                    axs[iens,ic1+1].errorbar(plt_x,plt_y,plt_yerr,color=color)
                    axs[-1,ic1+1].errorbar(plt_x,plt_y,plt_yerr,color=colors[iens])
                    
                    
                    if j in ['j-;conn']:
                        Q2cut=fit_Q2_dipole_Q2cut
                        doFit=yu.doFit_dipole
                        fitfunc=yu.fitfunc_dipole
                    else:
                        Q2cut=fit_Q2_const_Q2cut
                        doFit=yu.doFit_const
                        fitfunc=yu.fitfunc_const
                    inds=[i for i,Q2 in enumerate(Q2s) if Q2<=Q2cut]
                    Q2s=Q2s[inds]
                    ys=ys[:,inds]
                
                    pars_jk,chi2_jk,Ndof=doFit(Q2s,ys,corrQ=False) if doFit!=yu.doFit_const else doFit(ys,corrQ=False) 
                    
                    xs_plt=np.arange(0,Q2cut+1e-5,0.05)
                    y_plt=np.array([fitfunc(xs_plt,*pars) for pars in pars_jk])
                    mean,err=yu.jackme(y_plt*Z)
                    plt_x=xs_plt*xunit; plt_y=mean*yunit; plt_yerr=err*yunit
                    
                    axs[iens,ic1+1].fill_between(plt_x,plt_y-plt_yerr,plt_y+plt_yerr,color=color,alpha=0.2,label=yu.un2str(plt_y[0],plt_yerr[0]))
                    axs[iens,ic1+1].legend()
                    
                    axs[iens,-1].errorbar(ic1,plt_y[0],plt_yerr[0],color=color,label=yu.un2str(plt_y[0],plt_yerr[0]))
                    axs[iens,-1].set_xlim([-5,5])
                    axs[iens,-1].legend()
                    
                    if c1=='unequal' and j in ['j+;conn']+js_disc:
                        t=y_plt[:,0]
                        ttej_key2bare_B20at0[(*tftcphy,ens,j)]=t
                    if c1=='all' and j=='j-;conn':
                        factor=ens2RCs_np_me[ens][f'Zqq(mu=nu)']/ens2RCs_np_me[ens][f'Zqq(mu!=nu)']
                        t=y_plt[:,0]*factor
                        ttej_key2bare_B20at0[(*tftcphy,ens,j)]=t

            yu.finalizePlot(f'{folder}/Q2fit_B20/{tftcphy}/{j}',closeQ=True,mkdirQ=True)
            yu.finalizePlot(closeQ=True)
            
    return ttej_key2bare_B20at0

ttej_key2bare_B20at0=yu.load_pkl_reg(f'{folder}/ttej_key2bare_B20at0')
if ttej_key2bare_B20at0 is None:
    ttej_key2bare_B20at0=run()
    yu.save_pkl_reg(f'{folder}/ttej_key2bare_B20at0',ttej_key2bare_B20at0)

# xJ

In [11]:
key2bare={}
for ens in enss:
    for j in js_conn:
        key2bare[(ens,j)]=ttej_key2bare_A20at0[(*tftcphy_A20_conn,ens,j)]
    for j in js_discq:
        key2bare[(ens,j)]=ttej_key2bare_A20at0[(*tftcphy_A20_discq,ens,j)]
    for j in js_gluon:
        nst=int(j.split('stout')[-1])
        key2bare[(ens,j)]=ttej_key2bare_A20at0[(*nst2tftcphy_A20_discg[nst],ens,j)]
yum.extendBare_avgx(key2bare)
key2phy_A20_pre=yum.bareRC2phy_avgx_np(key2bare,ens2RCs_np)
        
key2bare={}
for ens in enss:
    for j in js_conn:
        key2bare[(ens,j)]=ttej_key2bare_B20at0[(*tftcphy_A20_conn,ens,j)]
    for j in js_discq:
        key2bare[(ens,j)]=ttej_key2bare_B20at0[(*tftcphy_A20_discq,ens,j)]
    for j in js_gluon:
        nst=int(j.split('stout')[-1])
        key2bare[(ens,j)]=ttej_key2bare_B20at0[(*nst2tftcphy_A20_discg[nst],ens,j)]
yum.extendBare_avgx(key2bare)
key2phy_B20_pre=yum.bareRC2phy_avgx_np(key2bare,ens2RCs_np)

In [12]:
stouts=stouts_jointlinear

key2phy_A20={(ens,f'jg;stout{nst}'):key2phy_A20_pre[(ens,f'jg;stout{nst}')] for ens in enss for nst in stouts }
for nst in stouts:
    key2phy_A20[('a=#_const',f'jg;stout{nst}')]=key2phy_A20_pre[('a=#_const',f'jg;stout{nst}')]
    key2phy_A20[('a=#_linear',f'jg;stout{nst}')]=key2phy_A20_pre[('a=#_linear',f'jg;stout{nst}')]
    key2phy_A20[('a=#_MA',f'jg;stout{nst}')]=key2phy_A20_pre[('a=#_MA',f'jg;stout{nst}')]

y_jk=yu.superjackknife([np.transpose([key2phy_A20[(ens,f'jg;stout{nst}')] for nst in stouts]) for ens in enss])
def fitfunc(pars):
    return [pars[0]+pars[1+ist]*yu.ens2a[ens]**2 for ens in enss for ist,stout in enumerate(stouts)]
pars_jk,chi2_jk,Ndof,Nwarning=yu.jackfit(fitfunc,y_jk,pars0=[0.4]+[0]*len(stouts))
# print(yu.jackme_un2str(y_jk))
print(yu.jackme_un2str(pars_jk[:,0]),np.mean(chi2_jk)/Ndof,Ndof)

fig,axs=yu.getFigAxs(1,1,sharex=True)
ax=axs[0,0]
for ist,nst in enumerate(stouts):
    color=yu.colors8[ist%8]; fmt=yu.fmts8[ist%8]
    x=yum.lat_a2s_plt
    t=np.array([pars[0]+pars[1+ist]*x for pars in pars_jk])
    if nst==10:
        phy_A20_g=t
    
    mean,err=yu.jackme(t)
    x=yum.lat_a2s_plt; ymin=mean-err; ymax=mean+err
    # ax.plot(x,mean,color=color,linestyle='--',marker='')
    ax.fill_between(x, ymin, ymax, color=color, alpha=0.1)
    
    for iens,ens in enumerate(enss):
        t=key2phy_A20[(ens,f'jg;stout{nst}')]
        mean,err=yu.jackme(t)
        plt_x=yu.ens2a[ens]**2+0.001/20*ist; plt_y=mean; plt_yerr=err
        ax.errorbar(plt_x,plt_y,plt_yerr,color=color,label=nst if iens==0 else None)
        
    t=key2phy_A20[('a=#_linear',f'jg;stout{nst}')][:,0]
    # print([yu.jackme_un2str(key2phy_A20[(ens,f'jg;stout{nst}')]) for ens in enss])
    # print(yu.jackme_un2str(t))
    mean,err=yu.jackme(t)
    plt_x=-0.001/10*(ist+1); plt_y=mean; plt_yerr=err
    ax.errorbar(plt_x,plt_y,plt_yerr,color=color,fmt=fmt)

ax.set_xlim([-0.001,0.008])
ax.legend(ncols=2)

yu.finalizePlot(f'{folder}/jointLinear',closeQ=True)

0.355(58) 0.9690690793137933 14


In [13]:
nst=10
ens='a=#_final'
key2phy_A20=key2phy_A20_pre.copy()
for j in ['jv1']:
    key2phy_A20[(ens,j)]=key2phy_A20[('a=#_MA',j)]
for j in ['jv2', 'jv3']:
    key2phy_A20[(ens,j)]=key2phy_A20[('a=#_MA',j)]
for j in [f'jg;stout{nst}']:
    key2phy_A20[(ens,j)]=phy_A20_g
for j in [f'jq;stout{nst}']:
    key2phy_A20[(ens,j)]=key2phy_A20[('a=#_MA',j)]
    
for fla in yum.fla2iso.keys():
    key2phy_A20[(ens,f'j{fla};stout{nst}')]=np.sum([c*key2phy_A20[(ens,f'j{iso};stout{nst}' if iso=='q' else f'j{iso}')] for c,iso in yum.fla2iso[fla]],axis=0)

key2phy_A20[ens,f'jtot;stout{nst}']=key2phy_A20[ens,f'jq;stout{nst}']+key2phy_A20[ens,f'jg;stout{nst}']
key2phy_A20=yum.convert_key2phy_stout(key2phy_A20,10)

key2phy_B20=key2phy_B20_pre.copy()
for key in key2phy_B20_pre.keys():
    ens,j=key
    if ens=='a=#_MA':
        key2phy_B20[('a=#_final',j)]=key2phy_B20[(ens,j)]
key2phy_B20=yum.convert_key2phy_stout(key2phy_B20,10)

key2phy={}
for key in key2phy_A20.keys():
    if key not in key2phy_A20 or key not in key2phy_B20:
        continue
    key2phy[key]=(key2phy_A20[key]+key2phy_B20[key])/2
key2phy_J=key2phy

In [14]:
cases=['avgx','B20','J']
for case in cases:
    key2phy={'avgx':key2phy_A20,'B20':key2phy_B20,'J':key2phy_J}[case]

    list_dic=[{
        'key2phy':key2phy,
    }]
    fig,axs=yum.makePlot_a2dependence_avgx(list_dic,case=case,ce='final')
    # addexp(axs[0,0],axs[1,0])
    
    if case in ['avgx']:
        axs[1,0].set_ylim([-0.1,0.55])
    if case in ['B20']:
        axs[0,0].set_ylim([-0.3,0.3])
        axs[1,0].set_ylim([-0.19,0.19])

    yu.finalizePlot(f'{folder}/a2dep_{case}')